# DPD — Test de casos desde Excel (offline, sin BD)

Lee `tests/Days Past Due.xlsx`, corre la lógica de DPD (modos `join` y `cascade`) **directamente sobre los datos del Excel** y muestra el resultado por contrato/caso para validar contra `Resultado esperado del script`.

**No toca MySQL** — usa `compute_from_data()` de cada modo, que es la misma lógica que se va a usar contra la BD pero recibe listas de dicts en lugar de un cursor.

Orden de las celdas:
1. Instalar dependencias (solo la primera vez).
2. Cargar Excel y normalizar tipos.
3. Convertir a las estructuras que esperan los modos.
4. Correr `cascade` y `join`, armar DataFrame con DPD por cuota.
5. Cruzar con la hoja `Casos` y mostrar resultado por caso.
6. Resumen por contrato.
7. Export CSV.

In [ ]:
# Solo si faltan en el venv. Comentar después de la primera corrida.
%pip install -q pandas openpyxl

In [ ]:
import os
import sys
from datetime import date
from decimal import Decimal

import pandas as pd

# Asegurar que el paquete dpd/ es importable cuando el notebook está en la raíz del repo.
sys.path.insert(0, os.path.abspath("."))

from dpd.config import RunConfig
from dpd.modes import join_installment, cascade_fifo

# === Parámetros ===
EXCEL_PATH = "tests/Days Past Due.xlsx"
CALC_DATE = date(2026, 10, 3)        # los casos hablan de "corte de 3 de octubre"
PARTIAL_COUNTS = False               # solo aplica en cascade

# Cargar las 3 hojas
xls = pd.ExcelFile(EXCEL_PATH)
cases_df = pd.read_excel(xls, sheet_name="Casos")
sched_df = pd.read_excel(xls, sheet_name="scheduled_payments_installments")
pay_df = pd.read_excel(xls, sheet_name="Payment_Tape")

# --- Sanitización del Excel ---
# Si `borrower_installment_reference` en payment_tape está vacía pero hay una
# `Unnamed: NN` con los datos al final, asumimos que es por columnas mal alineadas
# en el Excel (caso real visto). Hacemos fallback automático y avisamos.
if pay_df["borrower_installment_reference"].isna().all():
    fallback_col = next(
        (c for c in pay_df.columns
         if c.startswith("Unnamed:") and pay_df[c].notna().any()),
        None,
    )
    if fallback_col:
        print(f"⚠ Payment_Tape.borrower_installment_reference está vacía — "
              f"uso '{fallback_col}' como referencia de cuota.")
        pay_df["borrower_installment_reference"] = pay_df[fallback_col]

# Normalizar tipos
sched_df["date"] = pd.to_datetime(sched_df["date"]).dt.date
pay_df["payment_date"] = pd.to_datetime(pay_df["payment_date"]).dt.date

# Las refs son numéricas en el Excel; las paso a string para los joins.
# Si la ref viene como float (porque tiene NaN en otras filas), trunco a entero antes.
def _ref_to_str(s: pd.Series) -> pd.Series:
    if pd.api.types.is_float_dtype(s):
        return s.where(s.isna(), s.astype("Int64")).astype(str)
    return s.astype(str)

sched_df["borrower_installment_reference"] = _ref_to_str(sched_df["borrower_installment_reference"])
pay_df["borrower_installment_reference"] = _ref_to_str(pay_df["borrower_installment_reference"])

# Asignar id sintético a cada cuota (en el Excel viene NaN) — necesario para hacer merge después.
sched_df = sched_df.reset_index(drop=True)
sched_df["id"] = sched_df.index + 1

print(f"Casos: {len(cases_df)} | Schedule: {len(sched_df)} | Payment_Tape: {len(pay_df)}")
print(f"Calc date: {CALC_DATE}")
print(f"Contratos en schedule: {sched_df['borrower_contract_id'].nunique()}")
print(f"Contratos en payment_tape: {pay_df['borrower_contract_id'].nunique()}")
print(f"Refs únicas en payment_tape: {sorted(pay_df['borrower_installment_reference'].dropna().unique())[:10]}")

In [ ]:
# Convierto los DataFrames a las listas de dicts que esperan los modos.
def installments_from_df(df):
    return [
        {
            "id": int(r["id"]),
            "borrower_contract_id": r["borrower_contract_id"],
            "borrower_installment_reference": r["borrower_installment_reference"],
            "installment_date": r["date"],
            "gross_amount": r.get("gross_amount", 0),
            "guarantee_amount": r.get("guarantee_amount", 0),
            "principal_amount": r.get("principal_amount", 0),
            "interest_amount": r.get("interest_amount", 0),
            "tax_amount": r.get("tax_amount", 0),
            "fee_amount": r.get("fee_amount", 0),
        }
        for _, r in df.iterrows()
    ]


def payments_from_df(df):
    return [
        {
            "borrower_contract_id": r["borrower_contract_id"],
            "borrower_installment_reference": r.get("borrower_installment_reference"),
            "payment_date": r["payment_date"],
            "total_payment": r.get("total_payment", 0),
        }
        for _, r in df.iterrows()
    ]


installments = installments_from_df(sched_df)
payments = payments_from_df(pay_df)

cfg_cascade = RunConfig(
    company_id=1, company_code="*", mode="cascade",
    partial_payment_counts=PARTIAL_COUNTS, calculation_date=CALC_DATE,
)
cfg_join = RunConfig(
    company_id=1, company_code="*", mode="join",
    partial_payment_counts=False, calculation_date=CALC_DATE,
)

cascade_results = list(cascade_fifo.compute_from_data(installments, payments, cfg_cascade))
join_results = list(join_installment.compute_from_data(installments, payments, cfg_join))

# Armo DataFrame con metadata + ambos modos lado a lado para auditar.
cascade_df = pd.DataFrame(cascade_results).rename(
    columns={"dpd_current": "dpd_cascade", "amount_in_arrears": "arrears_cascade"}
)
join_df = pd.DataFrame(join_results).rename(
    columns={"dpd_current": "dpd_join", "amount_in_arrears": "arrears_join"}
)

meta_cols = [
    "id", "borrower_contract_id", "borrower_installment_reference",
    "date", "gross_amount", "principal_amount", "interest_amount",
]
detail = (
    sched_df[meta_cols]
    .merge(cascade_df, on="id", how="left")
    .merge(join_df, on="id", how="left")
    .sort_values(["borrower_contract_id", "date"])
)

# total pagado por (contrato, ref) para auditar
paid_by_ref = (
    pay_df.groupby(["borrower_contract_id", "borrower_installment_reference"], as_index=False)
          .agg(total_paid_by_ref=("total_payment", "sum"))
)
detail = detail.merge(
    paid_by_ref, on=["borrower_contract_id", "borrower_installment_reference"], how="left"
)
detail["total_paid_by_ref"] = detail["total_paid_by_ref"].fillna(0)

print(f"Cuotas: {len(detail)}")
print(f"Cuotas con dpd_cascade > 0: {(detail['dpd_cascade'] > 0).sum()}")
print(f"Cuotas con dpd_join    > 0: {(detail['dpd_join']    > 0).sum()}")
detail.head(20)

In [ ]:
# Resultado por CASO — cruzo la hoja Casos contra el cómputo, mostrando descripción + esperado + calculado.
# Como `Resultado esperado del script` está en prosa, comparación visual; los DPD calculados van al lado.

# Renombro columnas de Casos para hacer el merge más limpio
cases = cases_df.rename(columns={
    "#": "caso",
    "Caso": "titulo",
    "Borrower contract id": "borrower_contract_id",
    "Descripción": "descripcion",
    "Resultado esperado del script": "esperado",
})

# Resumen por contrato (DPD del crédito = MAX entre cuotas)
per_contract = (
    detail.groupby("borrower_contract_id", as_index=False)
          .agg(
              dpd_cascade=("dpd_cascade", "max"),
              arrears_cascade=("arrears_cascade", "sum"),
              dpd_join=("dpd_join", "max"),
              arrears_join=("arrears_join", "sum"),
              cuotas=("id", "count"),
              cuotas_mora_cascade=("dpd_cascade", lambda s: (s > 0).sum()),
              cuotas_mora_join=("dpd_join", lambda s: (s > 0).sum()),
          )
)

cases_view = cases.merge(per_contract, on="borrower_contract_id", how="left")

# Mostrar columnas relevantes en orden razonable
cols = [
    "caso", "titulo", "borrower_contract_id",
    "dpd_cascade", "arrears_cascade",
    "dpd_join", "arrears_join",
    "cuotas", "cuotas_mora_cascade", "cuotas_mora_join",
    "descripcion", "esperado",
]
cases_view = cases_view[[c for c in cols if c in cases_view.columns]]

with pd.option_context("display.max_colwidth", 200, "display.width", 220):
    print("=== Resultado por caso ===")
    display(cases_view)
cases_view

In [ ]:
# Detalle por cuota para un caso específico — útil para depurar discrepancias.
CASO_DEBUG = 1   # número de caso (columna `#` en la hoja Casos)

contract = cases_view.loc[cases_view["caso"] == CASO_DEBUG, "borrower_contract_id"].iloc[0]
print(f"Caso {CASO_DEBUG} — contrato {contract}")
print(cases_view.loc[cases_view["caso"] == CASO_DEBUG, "descripcion"].iloc[0])
print(f"\nEsperado: {cases_view.loc[cases_view['caso'] == CASO_DEBUG, 'esperado'].iloc[0]}\n")

debug_cols = [
    "borrower_installment_reference", "date", "gross_amount",
    "total_paid_by_ref",
    "dpd_cascade", "arrears_cascade", "dpd_join", "arrears_join",
]
detail.loc[detail["borrower_contract_id"] == contract, debug_cols].sort_values("date")

In [ ]:
# Export a CSV — celda independiente.
# Tres archivos para que sea fácil revisar:
#   - dpd_detalle_cuotas.csv : cuota por cuota con DPD en cascade y join
#   - dpd_por_contrato.csv   : agregado por contrato
#   - dpd_por_caso.csv       : casos del Excel + DPD calculado, para revisar contra "esperado"
out_dir = os.path.abspath(".")

detail.to_csv(os.path.join(out_dir, "dpd_detalle_cuotas.csv"), index=False)
per_contract.to_csv(os.path.join(out_dir, "dpd_por_contrato.csv"), index=False)
cases_view.to_csv(os.path.join(out_dir, "dpd_por_caso.csv"), index=False)

print("Exportado en:", out_dir)
print(f"  dpd_detalle_cuotas.csv  ({len(detail)} filas)")
print(f"  dpd_por_contrato.csv    ({len(per_contract)} filas)")
print(f"  dpd_por_caso.csv        ({len(cases_view)} filas)")